# 导入需要的库

In [1]:
import pandas as pd
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from preprocess import gen_data_set, gen_model_input
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler

# 这一行负责从 preprocess.py 文件中加载 gen_model_input 函数
from preprocess import gen_data_set, gen_model_input 

## 处理数据

In [2]:
import random
import csv
import os
import pandas as pd
from tqdm import tqdm

# 源文件路径
# PWD: e:\自学代码\搜推算法\deepmatch\DeepMatch\examples
path = './Tenrec/QB-video.csv'

# 分割后的子文件存储路径

workspace = './Tenrec/ctr_task'

with open(path, 'r', newline='') as file:
    csvreader = csv.reader(file)
    a = next(csvreader)
    #print(a)
    i = j = 0
    for row in csvreader:
        # 每 10000 个就 j 加 1，然后就有一个新的文件名。
        if i % 1000000 == 0:
            j += 1
            print("完成第{}个100w数据".format(j-1))
#
        csv_path = os.path.join(workspace, 'QK_video1M.csv')
        user_id = int(row[0])
        if user_id < 1000017:
#         # 不存在此文件的时候，就创建
            if not os.path.exists(os.path.dirname(csv_path)):
                os.makedirs(os.path.dirname(csv_path))
                with open(csv_path, 'w', newline='') as file:
                    csvwriter = csv.writer(file)
                    csvwriter.writerow(row)
                i += 1
    #         # 存在的时候就往里面添加
            else:
                with open(csv_path, 'a', newline='') as file:
                    csvwriter = csv.writer(file)
                    csvwriter.writerow(row)
                i += 1

path = os.path.join(workspace, 'QK_video1M.csv')
source_data = pd.read_csv(path)
source_data.columns = ['user_id', 'item_id', 'click', 'follow', 'like', 'share', 'short_v', 'play_times', 'gender', 'age']
pos_data = source_data[source_data.click.isin([1])]
user_history = pos_data.groupby('user_id').item_id.apply(list).to_dict()
del pos_data
user_hist = {}
user_target = {}
for key, value in user_history.items():
    if len(value) <= 10:
        if len(value) > 1:
            user_hist[key] = value[:-1] + (10 - len(value[:-1])) * [0]
            user_target[key] = value[-1:]
        else:
            user_hist[key] = [0] * 10
            user_target[key] = value
    else:
        user_hist[key] = value[:10]
        user_target[key] = value[10:]
del_list = []
for key, value in tqdm(user_hist.items()):
    for v in value:
        del_list.append([key, v])

def del_data(s_data, user_hist, i):
    print('++++++++{}+++++++'.format(i))
    new = []
    for user, value in tqdm(user_hist.items()):
        if user > i * 10000 and user <= (i+1) * 10000:
            data = s_data[s_data['user_id'] == user]
            tmp_data = data[~data.item_id.isin(value)].values.tolist()
            new.extend(tmp_data)
    return new

max_len = 1100000
times = max_len // 10000
new_list = []

for i in range(times):
    print("+++++++++times{}+++++++++".format(i))
    data = source_data[(source_data['user_id'] > i * 10000) & (source_data['user_id'] <= (i+1) * 10000)]
    new = del_data(data, user_hist, i)
    new_list.extend(new)
    # if len(new_list) == 100000 or i == times - 1:
new_data = pd.DataFrame(new_list, columns=source_data.columns)

hist_1, hist_2, hist_3, hist_4, hist_5, hist_6, hist_7, hist_8, hist_9, hist_10 = {}, {}, {}, {}, {}, {}, {}, {}, {}, {}
for user, value in tqdm(user_hist.items()):
    hist_1[user] = value[0]
    hist_2[user] = value[1]
    hist_3[user] = value[2]
    hist_4[user] = value[3]
    hist_5[user] = value[4]
    hist_6[user] = value[5]
    hist_7[user] = value[6]
    hist_8[user] = value[7]
    hist_9[user] = value[8]
    hist_10[user] = value[9]

for i in range(1, 11):
    new_data['hist_{}'.format(i)] = new_data['user_id']
    # new_data['hist_{}'.format(i)] = new_data['hist_{}'.format(i)].map(hist)
new_data['hist_1'] = new_data['hist_1'].map(hist_1)
new_data['hist_2'] = new_data['hist_2'].map(hist_2)
new_data['hist_3'] = new_data['hist_3'].map(hist_3)
new_data['hist_4'] = new_data['hist_4'].map(hist_4)
new_data['hist_5'] = new_data['hist_5'].map(hist_5)
new_data['hist_6'] = new_data['hist_6'].map(hist_6)
new_data['hist_7'] = new_data['hist_7'].map(hist_7)
new_data['hist_8'] = new_data['hist_8'].map(hist_8)
new_data['hist_9'] = new_data['hist_9'].map(hist_9)
new_data['hist_10'] = new_data['hist_10'].map(hist_10)
new_data.to_csv('./ctr_data_1M.csv', header=True, index=False)



完成第0个100w数据
完成第1个100w数据
完成第2个100w数据
完成第3个100w数据
完成第4个100w数据
完成第5个100w数据
完成第6个100w数据
完成第7个100w数据
完成第8个100w数据
完成第9个100w数据
完成第10个100w数据
完成第11个100w数据
完成第12个100w数据
完成第13个100w数据
完成第14个100w数据
完成第15个100w数据
完成第16个100w数据
完成第17个100w数据
完成第18个100w数据
完成第19个100w数据
完成第20个100w数据
完成第21个100w数据
完成第22个100w数据
完成第23个100w数据
完成第24个100w数据
完成第25个100w数据
完成第26个100w数据
完成第27个100w数据
完成第28个100w数据
完成第29个100w数据
完成第30个100w数据
完成第31个100w数据
完成第32个100w数据
完成第33个100w数据
完成第34个100w数据
完成第35个100w数据
完成第36个100w数据
完成第37个100w数据
完成第38个100w数据
完成第39个100w数据
完成第40个100w数据
完成第41个100w数据
完成第42个100w数据
完成第43个100w数据
完成第44个100w数据
完成第45个100w数据
完成第46个100w数据
完成第47个100w数据
完成第48个100w数据
完成第49个100w数据
完成第50个100w数据
完成第51个100w数据
完成第52个100w数据
完成第53个100w数据
完成第54个100w数据
完成第55个100w数据
完成第56个100w数据
完成第57个100w数据
完成第58个100w数据
完成第59个100w数据
完成第60个100w数据
完成第61个100w数据
完成第62个100w数据
完成第63个100w数据
完成第64个100w数据


100%|██████████| 654/654 [00:00<00:00, 11404.58it/s]


+++++++++times0+++++++++
++++++++0+++++++


100%|██████████| 654/654 [00:00<00:00, 183876.85it/s]


+++++++++times1+++++++++
++++++++1+++++++


...

100%|██████████| 654/654 [00:00<?, ?it/s]


+++++++++times108+++++++++
++++++++108+++++++


100%|██████████| 654/654 [00:00<00:00, 652336.46it/s]


+++++++++times109+++++++++
++++++++109+++++++


100%|██████████| 654/654 [00:00<00:00, 524488.49it/s]

100%|██████████| 654/654 [00:00<00:00, 524488.49it/s]


# 读取数据

In [3]:
# data_path = "./"

# unames = ['user_id','gender','age','occupation','zip']
# user = pd.read_csv(data_path+'ml-1m/users.dat',sep='::',header=None,names=unames)
# rnames = ['user_id','movie_id','rating','timestamp']
# ratings = pd.read_csv(data_path+'ml-1m/ratings.dat',sep='::',header=None,names=rnames)
# mnames = ['movie_id','title','genres']
# movies = pd.read_csv(data_path+'ml-1m/movies.dat',sep='::',header=None,names=mnames,encoding="unicode_escape")
# movies['genres'] = list(map(lambda x: x.split('|')[0], movies['genres'].values))

# #pd.merge 函数在不指定 on 参数时，会自动寻找两个表中名称相同的列作为连接键（Key）。
# # 默认的合并方式是 Inner Join（内连接），即只保留两个表中都存在的 Key。
# data = pd.merge(pd.merge(ratings,movies),user)#.iloc[:10000]
# #例如Pandas 发现ratings表和movies表都有 movie_id 这一列，于是自动将其作为连接键。


# === 修改为读取 Tenrec 数据集 ===
# 假设数据已经通过 tenrec_gen_ctr.py 生成并保存在当前目录下，文件名为 ctr_data_1M.csv
# 如果没有生成，请确保先运行该脚本生成数据。
# 这里我们模拟读取该 CSV，并适配原 DSSM 代码的列名要求。

try:
    # 尝试读取 Tenrec 数据
    # 注意：根据 tenrec_gen_ctr.py，生成的列包含 user_id, item_id, click, gender, age ...
    data = pd.read_csv('./ctr_data_1M.csv')
    
    # 1. 字段重命名以适配原代码
    # item_id -> movie_id (偷懒做法，复用原代码逻辑)
    data.rename(columns={'item_id': 'movie_id'}, inplace=True)
    
    # 2. 补充缺失字段 (原 ML-1M 代码依赖 timestamp, genres 等)
    
    # timestamp: Tenrec 数据无时间戳，但一般负采样生成的数据在同一用户内相对有序。
    # 这里简单给一个递增序列作为伪时间戳，这对于序列切分很重要（preprocess.py 依赖它排序）
    data['timestamp'] = range(len(data))
    
    # genres: Tenrec 只提供了 item_id，没有提供物品类别。
    # 我们只能删掉模型里对 genres 的依赖，或者给一个默认值 "All"。
    # 为了简化改动，暂时给一个默认值，后续在 config 里把 genres 删掉。
    data['genres'] = 'General' 
    
    # occupation, zip: Tenrec 用户画像只有 gender, age。
    # 填充默认值或后续删除。
    data['occupation'] = 0
    data['zip'] = 0
    data['rating'] = data['click'] # 将点击行为视为评分/正反馈

    # 过滤掉未点击的数据用于序列构建?
    # 原 ML-1M 数据全是“交互过”的数据。Tenrec 数据包含 click=0/1。
    # DSSM 这种召回模型通常只学习正样本交互序列。
    data = data[data['click'] == 1].copy()

    print("Tenrec 数据加载成功")
    print(data.head())

except FileNotFoundError:
    print("错误：未找到 ctr_data_1M.csv 文件。请确保已运行 data_process 脚本生成数据。")


Tenrec 数据加载成功
    user_id  movie_id  click  follow  like  share short_v  play_times  gender  \
19     2138   1446867      1       0     0      0       1           3       0   
20     2138   1831290      1       0     0      0       0           4       0   
21     2138   1744473      1       0     0      0       0           2       0   
22     2138   1597321      1       0     0      0       0           2       0   
27     2138   1356663      1       0     0      0       0           2       0   

    age  ...   hist_6   hist_7   hist_8   hist_9  hist_10  timestamp   genres  \
19    0  ...  1524428  1514403  1428355  1452826  3300984         19  General   
20    0  ...  1524428  1514403  1428355  1452826  3300984         20  General   
21    0  ...  1524428  1514403  1428355  1452826  3300984         21  General   
22    0  ...  1524428  1514403  1428355  1452826  3300984         22  General   
27    0  ...  1524428  1514403  1428355  1452826  3300984         27  General   

    occupati

# 构建特征列，训练模型，导出embedding

In [4]:
#data = pd.read_csvdata = pd.read_csv("./movielens_sample.txt")
sparse_features = ["movie_id", "user_id",
                    "gender", "age", "occupation", "zip", "genres"]
SEQ_LEN = 50
negsample = 0

In [5]:
#data = pd.read_csvdata = pd.read_csv("./movielens_sample.txt")

# === 修改特征列表 ===
# 移除了 Tenrec 数据集缺失的 occupation, zip, genres
sparse_features = ["movie_id", "user_id", "gender", "age"]
SEQ_LEN = 10 # Tenrec 数据集预处理生成了10个历史行为
negsample = 0

# 1.Label Encoding
feature_max_idx = {}
for feature in sparse_features:
    lbe = LabelEncoder()
    data[feature] = lbe.fit_transform(data[feature]) + 1
    feature_max_idx[feature] = data[feature].max() + 1
    
    # 特殊处理：hist_1 到 hist_10 也是 movie_id，必须用同一个 LabelEncoder 转换
    if feature == "movie_id":
        for i in range(1, 11):
            col_name = f'hist_{i}'
            # 注意：历史行为里可能有填充的 0 或新 ID，这里简单起见，
            # 假设历史 ID 都在 movie_id 列里出现过（通常是成立的），或者遇到没见过的给默认值 0
            # 为了防止报错，先用 map 转换，未见过的填 0
            
            # 创建映射字典
            mapping = dict(zip(lbe.classes_, lbe.transform(lbe.classes_) + 1))
            data[col_name] = data[col_name].map(mapping).fillna(0).astype(int)

#按照user_id去重，保留性别、年龄等用户属性
user_profile = data[["user_id", "gender", "age"]].drop_duplicates('user_id')
item_profile = data[["movie_id"]].drop_duplicates('movie_id')

print('特征编码完成')
print(data[['movie_id', 'hist_1', 'hist_10']].head())
print('=======================================')

特征编码完成
    movie_id  hist_1  hist_10
19      9513       0        0
20     12834       0        0
21     12533       0        0
22     11728       0        0
27      3933       0        0


In [6]:
#把 user_id 设为 user_profile 这个 DataFrame 的索引（Index）。
user_profile.set_index("user_id", inplace=True)
print('设置用户ID为索引 (Set User ID as Index)')
print(user_profile.head())
print('=======================================')

#构建用户的原始历史行为序列
user_item_list = data.groupby("user_id")['movie_id'].apply(list)
print('构建用户-物品交互列表 (User-Item Interaction List)')
print(user_item_list.head())
print('=======================================')

设置用户ID为索引 (Set User ID as Index)
         gender  age
user_id             
1             1    1
2             2    4
3             2    5
4             1    1
5             2    4
构建用户-物品交互列表 (User-Item Interaction List)
user_id
1    [9513, 12834, 12533, 11728, 3933, 6425, 6576, ...
2    [4623, 7421, 2639, 1838, 4408, 19386, 15076, 1...
3    [3067, 4486, 9643, 3198, 12891, 662, 8551, 119...
4           [19121, 15081, 15842, 10340, 19318, 18763]
5                                [18730, 16431, 20658]
Name: movie_id, dtype: object


In [7]:
# 数据集划分
# 由于数据已经是处理好的样本，直接按 8:2 划分训练集和测试集
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(data, test_size=0.2, random_state=2022)

print('数据集切分 (Dataset Splitting)')
print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")
print('=======================================')

数据集切分 (Dataset Splitting)
Train size: 45650
Test size: 11413


In [9]:
# 生成模型输入 (Generate Model Input)
# 我们需要把 DataFrame 转换成 Keras 能够接受的字典形式
import numpy as np

def get_model_input(df):
    # 1. 提取基础特征
    model_input = {
        "user_id": df['user_id'].values,
        "movie_id": df['movie_id'].values,
        "gender": df['gender'].values,
        "age": df['age'].values,
    }
    
    # 2. 提取历史行为序列 (hist_1 ... hist_10)
    # 将这 10 列合并成一个 (N, 10) 的矩阵
    hist_cols = [f'hist_{i}' for i in range(1, 11)]
    history_matrix = df[hist_cols].values
    model_input['hist_movie_id'] = history_matrix
    
    # 3. 计算实际序列长度 (hist_len)
    # 统计每一行中不为 0 的元素个数
    model_input['hist_len'] = np.count_nonzero(history_matrix, axis=1)
    
    # 4. 提取标签 (Click)
    label = df['click'].values
    
    return model_input, label

train_model_input, train_label = get_model_input(train_data)
test_model_input, test_label = get_model_input(test_data)

print('生成模型输入完成')
print("Train Input Sample - user_id:", train_model_input['user_id'][:2])
print("Train Input Sample - hist_movie_id:", train_model_input['hist_movie_id'][:2])
print("Train Label Sample:", train_label[:5])
print('=======================================')

生成模型输入完成
Train Input Sample - user_id: [500  17]
Train Input Sample - hist_movie_id: [[19958     0     0 19932     0  4574 13326  6288  2272  8473]
 [ 2647     0     0 13345 12973 19727     0  3376 12602     0]]
Train Label Sample: [1 1 1 1 1]


In [10]:
# 2.Feature Config配置

embedding_dim = 32

#构建用户塔特征
user_feature_columns = [
    # 1). 稀疏特征 (Categorical Features)
    SparseFeat('user_id', feature_max_idx['user_id'], 16),
    SparseFeat("gender", feature_max_idx['gender'], 16),
    SparseFeat("age", feature_max_idx['age'], 16),
    
    # 2). 变长序列特征 (Sequence Features)
    # 注意 maxlen=10，对应 hist_1 到 hist_10
    VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                embedding_name="movie_id"), 
                                maxlen=10, combiner='mean', length_name='hist_len'),
]

#构建物品塔特征
item_feature_columns = [
    SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim),
]

print("特征配置完成")

特征配置完成


In [11]:
#4). 负采样配置 (Negative Sampler Config)
from collections import Counter

##统计训练集中每个电影出现的次数
train_counter = Counter(train_model_input['movie_id'])
#结果：{1: 500次, 2: 12次, 3: 10000次...}。这就是所谓的“频次直方图”。

##生成一个列表，索引是 movie_id，值是该 movie_id 出现的次数
item_count = [train_counter.get(i,0) for i in range(item_feature_columns[0].vocabulary_size)]
#转化成数组：模型（TensorFlow）不认识 Python 字典，它只认数组。
#逻辑：
#按照 movie_id 从 0 到 Max 的顺序，把频次排成一个长长的列表。
#get(i, 0)：如果某个 ID 没出现过，填 0。
#item_feature_columns[0].vocabulary_size：确保列表长度覆盖了所有可能的电影 ID。


'''
为什么要统计 item_count?
在进行采样 Softmax (Sampled Softmax) 时，
我们通常希望对热门物品进行降权，或者在计算概率时修正偏差（bias）。
item_count 提供了热门程度的信息，防止极其热门的物品主导了梯度的更新。
'''
#s(u,i)=ModelScore(u,i)−log(Prob(i))
#Prob(i) 是物品 i 出现的概率（频率）。
#−log(Prob(i)) 这一项的作用是：如果一个物品特别热门（概率大），我就要减掉一部分分非数。 这强迫模型去学习用户和物品真正的匹配度，排除掉“因为它热门所以你点了”的干扰因素。
#DeepMatch 的 sampledsoftmaxloss 内部实现了这个逻辑，而 item_count 就是用来计算这个 Prob(𝑖)的原材料。

'\n为什么要统计 item_count?\n在进行采样 Softmax (Sampled Softmax) 时，\n我们通常希望对热门物品进行降权，或者在计算概率时修正偏差（bias）。\nitem_count 提供了热门程度的信息，防止极其热门的物品主导了梯度的更新。\n'

In [12]:
train_counter

Counter({14125: 64,
         14121: 56,
         14331: 51,
         14324: 49,
         12419: 48,
         11717: 48,
         14647: 47,
         18767: 46,
         14815: 46,
         9183: 45,
         14286: 45,
         14872: 45,
         5891: 43,
         8551: 43,
         14636: 42,
         14381: 42,
         6658: 41,
         4727: 40,
         12930: 40,
         12115: 40,
         3687: 40,
         14758: 38,
         14109: 37,
         14756: 37,
         14160: 37,
         3887: 36,
         14320: 36,
         14233: 36,
         6328: 35,
         14103: 35,
         14314: 35,
         5534: 35,
         14598: 34,
         6271: 34,
         3683: 34,
         6977: 34,
         5560: 34,
         5225: 34,
         9351: 34,
         14900: 33,
         4623: 33,
         14123: 33,
         4408: 33,
         15562: 32,
         14344: 32,
         4443: 32,
         4640: 32,
         8142: 32,
         14638: 32,
         5056: 32,
         7435: 31,
  

In [13]:
item_count

[0,
 1,
 0,
 1,
 1,
 1,
 3,
 1,
 2,
 0,
 1,
 1,
 1,
 1,
 7,
 1,
 1,
 10,
 1,
 1,
 3,
 3,
 25,
 2,
 1,
 1,
 1,
 0,
 8,
 2,
 0,
 1,
 1,
 3,
 2,
 1,
 2,
 1,
 1,
 3,
 2,
 4,
 1,
 2,
 3,
 1,
 1,
 3,
 2,
 10,
 1,
 1,
 1,
 14,
 7,
 1,
 0,
 2,
 3,
 0,
 1,
 1,
 1,
 1,
 1,
 15,
 2,
 1,
 3,
 4,
 1,
 1,
 1,
 1,
 0,
 12,
 1,
 1,
 1,
 0,
 7,
 1,
 0,
 1,
 2,
 4,
 1,
 1,
 1,
 2,
 1,
 2,
 6,
 0,
 1,
 0,
 1,
 4,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 2,
 1,
 1,
 3,
 0,
 1,
 1,
 3,
 1,
 2,
 1,
 4,
 0,
 1,
 3,
 1,
 5,
 3,
 1,
 2,
 10,
 1,
 1,
 0,
 1,
 2,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 5,
 3,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 15,
 2,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 12,
 5,
 0,
 17,
 13,
 21,
 1,
 3,
 1,
 6,
 0,
 15,
 1,
 0,
 1,
 8,
 0,
 8,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 3,
 1,
 0,
 2,
 5,
 8,
 1,
 0,
 1,
 3,
 1,
 1,
 4,
 2,
 2,
 1,
 0,
 6,
 2,
 1,
 1,
 3,
 1,
 1,
 1,
 1,
 1,
 3,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 4,
 1,
 1,
 3,
 1,
 1,
 4,
 1,
 4,
 5,
 1,
 3,
 1,
 1,
 1,
 2,
 4,
 1,
 0,

In [14]:
## 定义采样器
sampler_config = NegativeSampler('inbatch',num_sampled=255,item_name="movie_id",item_count=item_count)
'''
'inbatch': 这是一个非常高效的训练策略。
    传统做法：
        对于每个 User，还要单独随机采样几千个他没看过的 Item 作为负样本。这会产生巨大的 IO 和计算开销。
    In-Batch 做法：
        假设一个 Batch 有 256 条数据 (User A - Movie A, User B - Movie B, ...)。
        对于 User A 来说，除了他自己看过的 Movie A 是正样本，
        Batch 里其他 255 个 User 看过的电影（Movie B, Movie C...）都可以天然作为负样本。
'''
'''
num_sampled=255: 在 Batch 内部采样多少个负样本。配合后面的 batch_size=256 使用。
'''

'\nnum_sampled=255: 在 Batch 内部采样多少个负样本。配合后面的 batch_size=256 使用。\n'

In [15]:
# 3.Define Model and train
#定义模型与训练 (Define Model and Train)
#这一阶段的目标是让模型学习如何将 User 和 Item 映射到同一个向量空间，使得用户和他感兴趣的物品距离更近。

import tensorflow as tf
#禁用了 TensorFlow 2.x 默认的 Eager Execution（动态图模式），强制使用 Graph Execution（静态图模式）。
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
'''
原因：
In-Batch Softmax 的特殊性：
    sampledsoftmaxloss 和内部的负采样逻辑涉及复杂的 Tensor 操作。
    在静态图模式下，TensorFlow 可以优化整个计算图，避免 Python 循环带来的开销，
    这对于大规模分类和负采样训练至关重要。
库的兼容性：
    DeepMatch/DeepCTR 的部分底层实现（尤其是早期的版本）是基于 TF1 的 Graph 逻辑编写的，
    关闭 Eager 模式能保证最大程度的兼容和稳定。
'''

'\n原因：\nIn-Batch Softmax 的特殊性：\n    sampledsoftmaxloss 和内部的负采样逻辑涉及复杂的 Tensor 操作。\n    在静态图模式下，TensorFlow 可以优化整个计算图，避免 Python 循环带来的开销，\n    这对于大规模分类和负采样训练至关重要。\n库的兼容性：\n    DeepMatch/DeepCTR 的部分底层实现（尤其是早期的版本）是基于 TF1 的 Graph 逻辑编写的，\n    关闭 Eager 模式能保证最大程度的兼容和稳定。\n'

##### 至此完成了准备数据（preprocess）、配置特征列（SparseFeat/VarLenSparseFeat）、配置负采样策略（NegativeSampler

#### 开始以构建并训练DSSM

In [16]:
# 4. Generate user features for testing and full item features for retrieval
#提取向量 (Generate Embeddings)
#这是**召回（Matching）模型与排序（Ranking）**模型最大的区别。 
# 训练结束后，我们需要把模型“拆”开，只要 User 塔和 Item 塔的输出，而不需要最后的 Loss 计算部分。

#准备预测数据
#用户侧使用测试集的用户数据
test_user_model_input = test_model_input
#物品侧使用全量数据库
# 移除了 'genres'key
all_item_model_input = {"movie_id": item_profile['movie_id'].values}

In [18]:
# 4. Generate user features for testing and full item features for retrieval
#提取向量 (Generate Embeddings)
#这是**召回（Matching）模型与排序（Ranking）**模型最大的区别。 
# 训练结束后，我们需要把模型“拆”开，只要 User 塔和 Item 塔的输出，而不需要最后的 Loss 计算部分。

#准备预测数据
#用户侧使用测试集的用户数据
test_user_model_input = test_model_input

#物品侧使用全量数据库
# 修正：item_profile 中只有 movie_id，且模型只使用了 movie_id 特征
# all_item_model_input = {"movie_id": item_profile['movie_id'].values,"genres": item_profile['genres'].values}
all_item_model_input = {"movie_id": item_profile['movie_id'].values}

In [19]:
#model 是一个完整的计算图。
#我们可以定义一个新的 user_embedding_model，它的输入和原模型一样，但输出截断在 User 塔的最后一层 (model.user_embedding)。Item 侧同理。
user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)
###意义：现在我们拥有了两个独立的模型：
#输入用户信息 -> 输出 32维 User 向量。
#输入物品信息 -> 输出 32维 Item 向量。

NameError: name 'model' is not defined

In [23]:
#批量推断 (Batch Inference)
#这步操作叫做 “离线向量化”（Offline Vectorization），是召回模型从“训练阶段”走向“服务/预测阶段”的分水岭
#核心目的：把训练好的神经网络，变成两个可以查表的 “向量矩阵”。
#训练时 (Training): 我们是 Pair-wise 的（成对输入）。模型就像一个严厉的教官，同时看着 User 和 Item，计算它们匹不匹配，然后调整它们各自的权重。 模型结构：User Input + Item Input -> Score -> Loss
#这步推断 (Inference): 我们现在要把教官这一对“拆散”。
    #User Side: 让所有用户排好队，一个个通过 User 塔，算出他们每个人的“兴趣向量” (user_embs)。
    #Item Side: 让全库所有电影排好队，一个个通过 Item 塔，算出它们每个电影的“特征向量” (item_embs)。

# 1. 计算用户向量
# Input: 测试集里的用户特征 (test_user_model_input)
# Action: 跑一遍 User Embedding Model (DSSM 的左塔)
# Output: user_embs (Shape: [用户数, 32])
# 含义: 每一行代表一个用户，即使这个用户很复杂（有画像、有几百条历史行为），现在都被压缩成了这 32 个数字。
user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)


# 2. 计算物品向量
# Input: 全库所有电影的特征 (all_item_model_input)
# Action: 跑一遍 Item Embedding Model (DSSM 的右塔)
# Output: item_embs (Shape: [电影总数, 32])
# 含义: 每一行代表一部电影，这 32 个数字浓缩了电影的风格、题材等信息。
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

print(user_embs.shape)
print(item_embs.shape)

(6040, 32)
(3706, 32)


e:\自学代码\搜推算法\deepmatch\venv\lib\site-packages\tensorflow\python\keras\engine\training.py:2455: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  warnings.warn('`Model.state_updates` will be removed in a future version. '


# 使用faiss进行ANN查找并评估结果

这几步是在进行全流程的实战演练：模拟一个真实的线上召回系统是如何工作的，并给它打分。

In [ ]:
#! pip install faiss-cpu

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [24]:
import numpy as np
import faiss
from tqdm import tqdm
from deepmatch.utils import recall_N

In [25]:
#1. 建库 (Indexing) —— "把图书馆建好"
#第一步：构建向量索引 (Build Index with Faiss)
#这一步相当于把所有商品都存入了向量数据库中，准备等待用户的查询。

# 1. 创建索引
# IndexFlatIP: "Flat"表示暴力搜索（不压缩，保证精度），"IP"代表 Inner Product（内积）。
# 在深度学习中，内积通常用于衡量向量相似度（类似于余弦相似度，如果向量归一化过的话）。
index = faiss.IndexFlatIP(embedding_dim)

## 2. 将物品向量加入索引
# item_embs: 之前算好的所有电影的 Embedding 矩阵，形状是 (电影总数, 32)。
# faiss.normalize_L2(item_embs)
index.add(item_embs)

In [26]:
#2. 检索 (Retrieval) —— "读者来借书"
#第二步：进行批量检索 (Batch Search)
#这里直接用所有测试集用户的向量去数据库里“搜”他们可能喜欢的电影。

# faiss.normalize_L2(user_embs)
## D: Distance (这里是内积得分)，形状 (用户数, 100)
# I: Index (物品的ID索引)，形状 (用户数, 100)
# search(user_embs, 100): 对每个用户向量，找出内积最大的 Top 100 个物品。
D, I = index.search(np.ascontiguousarray(user_embs), 100) # 将搜索结果从50改为100，以便后续计算Recall@100
#np.ascontiguousarray：Faiss 要求输入数据在内存中是连续的 C-order 数组，这是一个常见的工程细节。
#结果：对于第 i 个用户，I[i] 保存了推荐给他的 100 个电影在 item_profile 中的行号（索引）。

#注意“计算内积这件事，被外包给了 Faiss”
#index.search 内部在做什么？ 它在做疯狂的矩阵乘法（Matrix Multiplication）。
#数学上Scores=UserMatrix×ItemMatrix^T
#user_embs 是 U×K（用户数 × 32）
#item_embs 是 I×K（物品数 × 32）
#它们相乘得到 U×I 的分数矩阵。
#Faiss 的厉害之处： 如果让你写 for 循环去算，或者用普通的 numpy 算，面对百万级商品可能会很慢。 Faiss 用了极度优化的 C++ 代码、SIMD 指令集甚至 GPU 加速，在几毫秒内就能算完这些内积，并且顺便帮你排个序，只把最大的 100 个分数值（D）和对应的物品编号（I）吐给你。

In [29]:
D

array([[0.5030044 , 0.48093268, 0.46101317, ..., 0.3055786 , 0.30446455,
        0.30379763],
       [0.55936044, 0.5293887 , 0.52400726, ..., 0.4065881 , 0.4059934 ,
        0.40522677],
       [0.6390761 , 0.6325694 , 0.62814134, ..., 0.48598102, 0.48432323,
        0.48358393],
       ...,
       [0.26168245, 0.2421192 , 0.22563207, ..., 0.18797356, 0.18792018,
        0.1879016 ],
       [0.4776782 , 0.46691364, 0.4663713 , ..., 0.36394083, 0.3636343 ,
        0.36356264],
       [0.4276092 , 0.41133505, 0.40982443, ..., 0.30403745, 0.30401075,
        0.3038764 ]], dtype=float32)

In [30]:
I

array([[2812, 2504, 2502, ..., 1434,  208,  210],
       [ 214,   39,  860, ...,   84,  619,  248],
       [ 136,  830,  339, ...,   96,  203,   22],
       ...,
       [ 639,  533,  561, ...,   58,  573, 1199],
       [ 132,   48,   97, ...,  223,  373,  350],
       [ 160,  630, 1012, ...,  396,  229,  391]], dtype=int64)

In [ ]:
# 3. 评估 (Evaluation) —— "看看推得准不准"
# 第三步：评估效果 (Evaluation)

# 1. 准备真实标签字典
# key是用户ID，value是该用户在测试集中实际看过的电影ID列表
# 修改说明：由于使用了 train_test_split，我们需要从 test_data DataFrame 中聚合用户的点击记录
test_true_label = {}
for user, item in zip(test_data['user_id'], test_data['movie_id']):
    if user not in test_true_label:
        test_true_label[user] = []
    test_true_label[user].append(item)

s = []
hit = 0

In [28]:
# 2. 遍历每个用户进行对比
# 这里一定要注意：上一步 index.search(..., 100) 必须把 k 改为 100，否则下面的 loop 拿到 pred 只有 50 个元素，算不出 K=100 的指标

# 定义存储结果的容器
metrics = {
    50: {'recall': [], 'precision': [], 'ndcg': [], 'hit': 0, 'f1': []},
    100: {'recall': [], 'precision': [], 'ndcg': [], 'hit': 0, 'f1': []}
}

for i, uid in tqdm(enumerate(test_user_model_input['user_id'])):
    try:
        # I[i]得到的是所有的索引(0, 1, 2...)的列表，长度是我们在 search 时指定的 (这里假设改成了 100)
        # 我们需要把它还原成真实的 movie_id
        pred_full = [item_profile['movie_id'].values[x] for x in I[i]]
        
        # 获取用户真实看过的电影列表 (在这个数据集中，长度通常为 1)
        true_items = test_true_label[uid] 

        # 分别计算 @50 和 @100 的指标
        for K in [50, 100]:
            # 截取前 K 个推荐结果
            pred_k = pred_full[:K]
            
            # --- 1. 基础命中统计 (Intersection) ---
            # 预测对了几个？
            hits = set(pred_k) & set(true_items)
            hit_count = len(hits)

            # --- 2. Hit Rate ---
            if hit_count > 0:
                metrics[K]['hit'] += 1
            
            # --- 3. Recall (召回率) ---
            # 命中数 / 真实感兴趣的总数
            # 如果真实只看了 1 个，命中就是 1/1=1，不命中就是 0/1=0
            recall_val = hit_count / len(true_items)
            metrics[K]['recall'].append(recall_val)
            
            # --- 4. Precision (精确率) ---
            # 命中数 / 推荐列表长度 K
            # 推荐了 50 个，中了 1 个，精度就是 1/50 = 2%
            precision_val = hit_count / K
            metrics[K]['precision'].append(precision_val)
            
            # --- 5. F1-Score ---
            # 调和平均数
            if (precision_val + recall_val) > 0:
                f1_val = 2 * (precision_val * recall_val) / (precision_val + recall_val)
            else:
                f1_val = 0
            metrics[K]['f1'].append(f1_val)

            # --- 6. NDCG (归一化折损累计增益) ---
            # 位置越靠前，价值越大
            
            # (1) 计算 DCG (实际增益)
            dcg_k = 0
            for rank, item in enumerate(pred_k):
                if item in true_items:
                    # rank 是从 0 开始的 (0, 1, 2...)
                    # 公式通常是 log2(rank + 2)，即第一名是 log2(2)=1，第二名是 log2(3)=1.58...
                    dcg_k += 1.0 / np.log2(rank + 2)
            
            # (2) 计算 IDCG (理想最大增益)
            # 假设所有真实感兴趣的物品都排在最前面
            idcg_k = 0
            # 理想情况下，前 len(true_items) 个位置都应该是对的
            for rank in range(min(len(true_items), K)):
                idcg_k += 1.0 / np.log2(rank + 2)
            
            # (3) 归一化
            if idcg_k > 0:
                metrics[K]['ndcg'].append(dcg_k / idcg_k)
            else:
                metrics[K]['ndcg'].append(0)

    except Exception as e:
        print(f"Error at index {i}: {e}") # 打印出错信息

# 打印最终报告
print("-" * 30)
for K in [50, 100]:
    print(f"Metrics @ {K}:")
    print(f"  Recall   : {np.mean(metrics[K]['recall']):.4f}")
    print(f"  Precision: {np.mean(metrics[K]['precision']):.4f}")
    print(f"  NDCG     : {np.mean(metrics[K]['ndcg']):.4f}")
    print(f"  F1-Score : {np.mean(metrics[K]['f1']):.4f}")
    print(f"  Hit Rate : {metrics[K]['hit'] / len(test_user_model_input['user_id']):.4f}")
    print("-" * 30)

0it [00:00, ?it/s]

6040it [00:00, 6938.08it/s]

------------------------------
Metrics @ 50:
  Recall   : 0.3581
  Precision: 0.0072
  NDCG     : 0.1113
  F1-Score : 0.0140
  Hit Rate : 0.3581
------------------------------
Metrics @ 100:
  Recall   : 0.4892
  Precision: 0.0049
  NDCG     : 0.1326
  F1-Score : 0.0097
  Hit Rate : 0.4892
------------------------------
